# Overlapping Data Transfers and Computation - MPI Edition

In this section we extend the MPI version of the 2D stencil with an overlap strategy that hides halo exchanges behind useful GPU work using CUDA streams and nonblocking MPI.
We follow the same approach as discussed in [09-overlap](./09-overlap.ipynb) but adapt it for our distributed memory parallel version.

As before, we use three dedicated streams which we add to the `Patch` struct:
* `topStream`: computes last inner row
* `bottomStream`: computes first inner row
* `bulkStream`: computes remaining interior rows

The updated approach is slightly more involved since the communication is not comprised of a send and a receive operations (was a one-sided push before):
1) Start asynchronous receives for top/ bottom halos.
2) Launch edge compute kernels on top/ bottom streams.
3) Launch bulk kernel on bulk stream.
4) Synchronize with the top/ bottom streams and start asynchronous sends of produced halo rows as soon as they become available.
5) Wait for bulk stream and MPI requests.
6) Swap buffers.

### 1. Start Receives

```cpp
MPI_Request reqs[4] = { MPI_REQUEST_NULL, MPI_REQUEST_NULL, MPI_REQUEST_NULL, MPI_REQUEST_NULL };

if (rank > 0)
  MPI_Irecv(&patch.d_localUNew[0 * patch.localNumCellsX], patch.localNumCellsX, MPI_DOUBLE, rank - 1, 0, MPI_COMM_WORLD, &reqs[0]);

if (rank < numRanks - 1)
  MPI_Irecv(&patch.d_localUNew[(patch.localNumCellsY - 1) * patch.localNumCellsX], patch.localNumCellsX, MPI_DOUBLE, rank + 1, 0, MPI_COMM_WORLD, &reqs[2]);
```


### 2. Compute Edge Rows

```cpp
stencil2D<<<ceilingDivide(patch.localNumCellsX - 2, 256), 256, 0, patch.bottomStream>>>(
    patch.d_localU, patch.d_localUNew,
    1, patch.localNumCellsX - 1, 
    1, 2,
    patch.localNumCellsX);

stencil2D<<<ceilingDivide(patch.localNumCellsX - 2, 256), 256, 0, patch.topStream>>>(
    patch.d_localU, patch.d_localUNew,
    1, patch.localNumCellsX - 1,
    patch.localNumCellsY - 2, patch.localNumCellsY - 1,
    patch.localNumCellsX);
```


### 3. Compute Bulk

```cpp
stencil2D<<<patch.gridSize, patch.blockSize, 0, patch.bulkStream>>>(
    patch.d_localU, patch.d_localUNew,
    1, patch.localNumCellsX - 1,
    2, patch.localNumCellsY - 2,
    patch.localNumCellsX);
```


### 4. Wait and Send

```cpp
cudaStreamSynchronize(patch.bottomStream);
if (rank > 0)
  MPI_Isend(&patch.d_localUNew[1 * patch.localNumCellsX], patch.localNumCellsX, MPI_DOUBLE, rank - 1, 0, MPI_COMM_WORLD, &reqs[1]);

cudaStreamSynchronize(patch.topStream);
if (rank < numRanks - 1)
  MPI_Isend(&patch.d_localUNew[(patch.localNumCellsY - 2) * patch.localNumCellsX], patch.localNumCellsX, MPI_DOUBLE, rank + 1, 0, MPI_COMM_WORLD, &reqs[3]);
```


### 5. Wait for the Rest

```cpp
MPI_Waitall(4, reqs, MPI_STATUSES_IGNORE);

cudaStreamSynchronize(patch.bulkStream);
```


### 6. Swap

```cpp
std::swap(patch.d_localU, patch.d_localUNew);
```


## Exercise

Choose one of the difficulty levels below to tailor the exercise to your preferences.
Each level provides a different starting point implementation to be copied into [stencil-2d.cu](../src/12-overlap/stencil-2d.cu).
Use it for your implementation and follow the steps outlined above.

Below the difficulty level descriptions, there are cells for compiling, executing and profiling the your solution.

### Level Hard

Create and empty file [stencil-2d.cu](../src/12-overlap/stencil-2d.cu) and copy one of the previous code versions into it (your work or a solution).

In [ ]:
!touch ../src/12-overlap/stencil-2d.cu

### Level Medium

[stencil-2d-medium.cu](../src/12-overlap/stencil-2d-medium.cu) contains a partial solution with TODOs ranging from straight-forward to complex.
Copy the provided code into the working file [stencil-2d.cu](../src/12-overlap/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
!cp ../src/12-overlap/stencil-2d-medium.cu ../src/12-overlap/stencil-2d.cu

### Level Easier

[stencil-2d-easier.cu](../src/12-overlap/stencil-2d-easier.cu) contains a partial solution with TODOs.
The solution is further progressed than the level medium version, and the complexity of the TODOs is limited.
Copy the provided code into the working file [stencil-2d.cu](../src/12-overlap/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
!cp ../src/12-overlap/stencil-2d-easier.cu ../src/12-overlap/stencil-2d.cu

### Possible Solution

[stencil-2d-solution.cu](../src/12-overlap/stencil-2d-solution.cu) contains a possible solution for this exercise.
Copy it into the working file [stencil-2d.cu](../src/12-overlap/stencil-2d.cu) with the cell below.

In [ ]:
!cp ../src/12-overlap/stencil-2d-solution.cu ../src/12-overlap/stencil-2d.cu

### Compilation, Execution and Profiling

The new code version is available in [12-overlap/stencil-2d.cu](../src/12-overlap/stencil-2d.cu) (after creating it with one of the commands above).
It can be compiled and executed with the following cells.

In [ ]:
!nvcc -ccbin=mpic++ -O3 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_86,code=sm_86 -o ../build/12-overlap ../src/12-overlap/stencil-2d.cu

The next cell produces output for a small grid.
Visualize the output using the [visualize](./99-visualize.ipynb) notebook after executing the application.

By default, it runs 2 ranks on a single GPU.
Use this to verify that your application works as intended.

In [ ]:
!mpirun -n 2 ../build/12-overlap 256 64 2 2000 100

The next cell produces no output and runs a larger grid.
Use it for performance evaluation.

In [ ]:
!mpirun -n 2 ../build/12-overlap $((32 * 1024)) 256 2 8 0

Instead of running on the local **single A40** GPU, we can also submit a batch job running on **multiple A100** GPUs.
Feel free to tune the number of GPUs (both in the `--gres=gpu:a100:NGPU` and in the `mpirun -n NGPU`) to anything between one and eight GPUs.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/12-overlap.out --error=../output/12-overlap.err \
    --wrap="mpirun -n 2 ../build/12-overlap $((32 * 1024)) 256 2 8 0"

cat ../output/12-overlap.out

The next cell performs profiling with Nsight Systems by submitting a batch job.
Feel free to tune the number of GPUs (both in the `--gres=gpu:a100:NGPU` and in the `mpirun -n NGPU`) to anything between one and eight GPUs.

The profile is then available at [profiles/12-overlap.nsys-rep](../profiles/12-overlap.nsys-rep) and can be downloaded by **shift + right-clicking** the link, by clicking the link with the **middle mouse button**, or using the JupyterHub file tree.

After downloading it, open it up locally to visualize the run-time behavior of your application.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/12-overlap-nsys.out --error=../output/12-overlap-nsys.err \
    --wrap="nsys profile --stats=true --force-overwrite=true \
        -o ../profiles/12-overlap \
        mpirun -n 2 \
            ../build/12-overlap $((32 * 1024)) 256 2 8 0"

cat ../output/12-overlap-nsys.out

## Next Step

The next step is learning about an alternative to MPI - NVSHMEM - in [13](./13-nvshmem.ipynb).